In [ ]:
# Thằng Hoan
import math
from pathlib import Path
from IPython.display import IFrame, display

# =========================
# 1) Thuật toán (Python) — chỉ dùng math
# =========================
EARTH_RADIUS_M = 6_371_000

def clamp_radius(value) -> float:
    """Ép bán kính không âm. Sai kiểu/NaN/Inf -> 0."""
    try:
        v = float(value)
    except (TypeError, ValueError):
        return 0.0
    if math.isnan(v) or math.isinf(v):
        return 0.0
    return max(0.0, v)

def destination_point(lat: float, lon: float, bearing_deg: float, distance_m: float, R: float = EARTH_RADIUS_M):
    """
    Destination point on sphere:
      lat/lon: degrees
      bearing_deg: degrees
      distance_m: meters
    return (lat2, lon2) degrees
    """
    phi1 = math.radians(lat)
    lam1 = math.radians(lon)
    theta = math.radians(bearing_deg)
    delta = distance_m / R

    sin_phi1, cos_phi1 = math.sin(phi1), math.cos(phi1)
    sin_delta, cos_delta = math.sin(delta), math.cos(delta)

    sin_phi2 = sin_phi1 * cos_delta + cos_phi1 * sin_delta * math.cos(theta)
    phi2 = math.asin(sin_phi2)

    y = math.sin(theta) * sin_delta * cos_phi1
    x = cos_delta - sin_phi1 * math.sin(phi2)
    lam2 = lam1 + math.atan2(y, x)

    return (math.degrees(phi2), math.degrees(lam2))

def build_ring(lat: float, lon: float, radius_m: float, steps: int = 160):
    """Tạo polygon xấp xỉ vòng tròn quanh (lat, lon)."""
    r = clamp_radius(radius_m)
    steps = max(12, int(steps))
    pts = []
    for i in range(steps + 1):
        bearing = (i / steps) * 360.0
        pts.append(destination_point(lat, lon, bearing, r))
    return pts

# =========================
# 3) Gọi map.html để hiển thị
# =========================
display(IFrame(src="/files/map.html", width="100%", height=650))

In [3]:
# Thằng Minh
# =========================
# 1) Haversine Distance
# =========================
def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371000  # bán kính trái đất (m)

    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)

    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)

    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlambda/2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))

    return R * c


# =========================
# 2) Tọa độ nhà bạn
# =========================
user_lat = 10.808
user_lon = 106.563

radius = 1500  # bán kính 1.5 km


# =========================
# 3) Dữ liệu trạm test
# =========================
stations = [
    {"name": "Trạm Sạc 1", "lat": 10.809, "lon": 106.564},
    {"name": "Trạm Sạc 2", "lat": 10.812, "lon": 106.570},
    {"name": "Trạm Sạc 3", "lat": 10.800, "lon": 106.550},
    {"name": "Trạm Sạc 4", "lat": 10.820, "lon": 106.600}
]


# =========================
# 4) Lọc trạm trong bán kính
# =========================
nearby = []

for s in stations:
    d = haversine_distance(user_lat, user_lon, s["lat"], s["lon"])

    if d <= radius:
        nearby.append({
            "name": s["name"],
            "distance_m": round(d,2),
            "lat": s["lat"],
            "lon": s["lon"]
        })


# =========================
# 5) Kết quả
# =========================
print("Các trạm trong bán kính", radius, "m:")

for s in nearby:
    print(f"{s['name']} - {s['distance_m']} m")

Các trạm trong bán kính 1500 m:
Trạm Sạc 1 - 155.86 m
Trạm Sạc 2 - 884.52 m


In [4]:
# Thằng Huy - GIS Chức năng 3: 2 Đường đi ngắn nhất đến trạm sạc
import math, json, urllib.request

ORS_API_KEY = "eyJvcmciOiI1YjNjZTM1OTc4NTExMTAwMDFjZjYyNDgiLCJpZCI6IjMyZGIyMGFlMjkwYTRjYmQ5MTVkYmU4NWM1YWU3MzFjIiwiaCI6Im11cm11cjY0In0="

# ── 1) Tìm trạm gần nhất ───────────────────────────────────────
def find_nearest(user_lat, user_lon, stations, radius_m=5000):
    result = []
    for s in stations:
        phi1, phi2 = math.radians(user_lat), math.radians(s["lat"])
        a = math.sin((phi2-phi1)/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(math.radians(s["lon"]-user_lon)/2)**2
        d = 6371000 * 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))
        if d <= radius_m:
            result.append({**s, "distance_m": round(d, 2)})
    return sorted(result, key=lambda x: x["distance_m"])[0] if result else None

# ── 2) Lấy 2 đường từ ORS ──────────────────────────────────────
def get_two_routes(user_lat, user_lon, to_lat, to_lon):
    body = json.dumps({
        "coordinates": [[user_lon, user_lat], [to_lon, to_lat]],
        "alternative_routes": {"target_count": 2, "weight_factor": 1.6, "share_factor": 0.6}
    }).encode()
    req = urllib.request.Request(
        "https://api.openrouteservice.org/v2/directions/driving-car/geojson",
        data=body,
        headers={"Authorization": ORS_API_KEY, "Content-Type": "application/json"},
        method="POST"
    )
    with urllib.request.urlopen(req, timeout=15) as r:
        features = json.loads(r.read())["features"]
    return [{"distance_m": round(f["properties"]["summary"]["distance"]),
             "duration_s": round(f["properties"]["summary"]["duration"])}
            for f in features[:2]]

# ── CHẠY ───────────────────────────────────────────────────────
user_lat, user_lon = 10.973, 106.900  
radius_m = 5000

# Thay bằng query PostgreSQL thực tế
stations = [
    {"name": "Trạm Sạc Vinfast - Vincom Biên Hòa", "lat": 10.9741, "lon": 106.8986},
    {"name": "Trạm Sạc Evgo - Lottemart",           "lat": 10.9680, "lon": 106.8830},
    {"name": "Trạm Sạc EVN - Khu CN Amata",         "lat": 10.9590, "lon": 106.8920},
    {"name": "Trạm Sạc ChargePoint - QL1A",         "lat": 10.9820, "lon": 106.8760},
    {"name": "Trạm Sạc Tesla - TTTM Go!",           "lat": 10.9500, "lon": 106.8650},
]

target = find_nearest(user_lat, user_lon, stations, radius_m)
if not target:
    print("❌ Không có trạm nào trong bán kính", radius_m, "m")
else:
    routes = get_two_routes(user_lat, user_lon, target["lat"], target["lon"])
    print(f"🎯 Trạm gần nhất: {target['name']} ({target['distance_m']} m)")
    for i, r in enumerate(routes):
        d = f"{r['distance_m']/1000:.2f} km" if r['distance_m'] >= 1000 else f"{r['distance_m']} m"
        t = f"{r['duration_s']//60} phút {r['duration_s']%60}s"
        print(f"  Đường {i+1}: {d} / {t}")

🎯 Trạm gần nhất: Trạm Sạc Vinfast - Vincom Biên Hòa (195.75 m)
  Đường 1: 759 m / 2 phút 50s
  Đường 2: 1.81 km / 3 phút 11s
